<a href="https://colab.research.google.com/github/GretelKMendez/Tareas-Mac-IA/blob/main/Netflix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Identificación del Problema
Tipo de problema: Clasificación Multiclase.
Aunque el dataset parece prestarse para análisis descriptivo, un problema relevante es predecir la Clasificación de Madurez (rating) (ej. PG-13, TV-MA, R) en función de las características del contenido. Esto es útil para automatizar la categorización de nuevos títulos que entran a la plataforma.

2. Variable Objetivo
Variable objetivo: rating.
Esta variable indica el público objetivo del contenido y es fundamental para los algoritmos de recomendación y control parental.

3. Selección de Variables (Justificación)
No todas las columnas son útiles para un modelo predictivo:

Variables a utilizar:

type: Diferencia entre película o serie, lo cual influye en el rating.

release_year: Las tendencias de clasificación han cambiado con las décadas.

duration: La longitud puede estar correlacionada con el tipo de contenido y su madurez.

listed_in: El género (Terror, Infantil, Documental) es el predictor más fuerte del rating.

Variables a descartar:

show_id: Identificador único sin valor estadístico.

title: Demasiada cardinalidad (un nombre único por fila).

date_added: La fecha de subida a Netflix no suele determinar la clasificación de madurez original.

description, cast, director: Requieren procesamiento de lenguaje natural avanzado (NLP) complejo; para este modelo base, se omiten para evitar sobreajuste.

4. Propuesta de Modelo
Modelo elegido: Bosques Aleatorios (Random Forest Classification).

Justificación:

Robustez: Al ser un conjunto de árboles de decisión, maneja mejor el ruido y evita el sobreajuste que un solo árbol podría tener.

Variables Categóricas: Este modelo funciona excepcionalmente bien con variables como "Género" o "País" una vez codificadas, capturando interacciones no lineales entre ellas (ej. un "Documental" de "1970" tiene un perfil de rating distinto a una "Serie de Terror" de "2023").



In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Cargar dataset
df = pd.read_csv('netflix_titles.csv')

# 1. Tratamiento de nulos
# El rating tiene pocos nulos, los llenaremos con la moda (el valor más común)
df['rating'] = df['rating'].fillna(df['rating'].mode()[0])

# 2. Ingeniería de variables: Simplificar la duración
# '80 min' -> 80 | '2 Seasons' -> 2
df['duration_num'] = df['duration'].apply(lambda x: int(x.split(' ')[0]) if pd.notnull(x) else 0)

# 3. Selección de columnas relevantes
features = ['type', 'release_year', 'duration_num', 'listed_in']
X = df[features]
y = df['rating']

# 4. Codificación de variables categóricas (Convertir texto a números)
le = LabelEncoder()
X['type'] = le.fit_transform(X['type'])
X['listed_in'] = le.fit_transform(X['listed_in'])

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(df)

     show_id     type                  title         director  \
0         s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1         s2  TV Show          Blood & Water              NaN   
2         s3  TV Show              Ganglands  Julien Leclercq   
3         s4  TV Show  Jailbirds New Orleans              NaN   
4         s5  TV Show           Kota Factory              NaN   
...      ...      ...                    ...              ...   
8802   s8803    Movie                 Zodiac    David Fincher   
8803   s8804  TV Show            Zombie Dumb              NaN   
8804   s8805    Movie             Zombieland  Ruben Fleischer   
8805   s8806    Movie                   Zoom     Peter Hewitt   
8806   s8807    Movie                 Zubaan      Mozez Singh   

                                                   cast        country  \
0                                                   NaN  United States   
1     Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa 

/tmp/ipykernel_4484/3143314646.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['type'] = le.fit_transform(X['type'])
/tmp/ipykernel_4484/3143314646.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['listed_in'] = le.fit_transform(X['listed_in'])


 Accuracy: Si el resultado es 0.55, significa que el modelo acierta el rating el 55% de las veces. Dado que hay muchas categorías de rating (TV-MA, TV-14, R, etc.), es un desempeño base aceptable, pero indica que el rating depende mucho de la description (texto) que no usamos.

In [2]:
# Inicializar el modelo
# Usamos 100 árboles para un balance entre precisión y velocidad
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Entrenar el modelo
model.fit(X_train, y_train)

# Realizar predicciones
y_pred = model.predict(X_test)

# Comparación de resultados
print(f"Precisión General (Accuracy): {accuracy_score(y_test, y_pred):.2f}")
print("\nInforme de Clasificación:")
print(classification_report(y_test, y_pred))

Precisión General (Accuracy): 0.49

Informe de Clasificación:
              precision    recall  f1-score   support

      66 min       0.00      0.00      0.00         1
           G       0.60      0.25      0.35        12
          NR       0.07      0.06      0.07        16
          PG       0.60      0.60      0.60        62
       PG-13       0.34      0.36      0.35        87
           R       0.42      0.37      0.39       163
       TV-14       0.43      0.45      0.44       414
        TV-G       0.11      0.07      0.08        43
       TV-MA       0.60      0.66      0.63       662
       TV-PG       0.30      0.21      0.24       185
        TV-Y       0.59      0.65      0.62        52
       TV-Y7       0.48      0.49      0.48        65
          UR       0.00      0.00      0.00         0

    accuracy                           0.49      1762
   macro avg       0.35      0.32      0.33      1762
weighted avg       0.48      0.49      0.48      1762



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [3]:
#Interpretación de variables: Al usar Random Forest, podemos ver qué característica fue más importante:
importances = pd.Series(model.feature_importances_, index=features)
print(importances.sort_values(ascending=False))

listed_in       0.437617
duration_num    0.345155
release_year    0.208790
type            0.008438
dtype: float64


con listed_in como primera opcion, confirmamos que el género es el principal determinante de si una película es para adultos o niños.